# Logging location demo (bluebird-gymnasium)

This notebook exercises the **bluebird-dt logging path** through a gymnasium environment,
so you can confirm that scenario logs are written to the new per-user data location
(`bluebird-scenario-logs/bluebird_dt`) rather than inside the installed package.

By default gymnasium envs run with logging switched off
(`save_log_to_file=False`, `autosave_interval=None`, `simulation_log_config['save_simulation']=False`).
Here we turn `save_simulation` **on** so that, when an episode finishes, the env calls
`save_simulation_logs()` -> `Simulator.save()`, writing a replay `.tar.gz` to `LOG_DIR`.

In [ ]:
import datetime
import os

from bluebird_dt.utility.paths import LOG_DIR, LOG_DIR_ENV_VAR
from bluebird_gymnasium.envs import SectorIEnv

## 1. Where will logs go?

`LOG_DIR` is resolved once at import. It is the per-user data directory
(overridable via the `BLUEBIRD_LOG_DIR` environment variable). We snapshot the
existing `.tar.gz` archives so we can tell which file this run produces.

In [ ]:
print('BLUEBIRD_LOG_DIR override:', os.environ.get(LOG_DIR_ENV_VAR) or '(not set)')
print('LOG_DIR             :', LOG_DIR)

before = set()
if os.path.isdir(LOG_DIR):
    before = {f for f in os.listdir(LOG_DIR) if f.endswith('.tar.gz')}
print('existing .tar.gz archives:', len(before))

## 2. Build a SectorI env with logging enabled

Minimal config (mirrors `simple_demo.ipynb`) plus `simulation_log_config` to switch on
saving and give the log a recognisable suffix.

In [ ]:
config = SectorIEnv.get_default_env_config()
config.view_config = {'type': 'decentralized', 'decentralized_params': {}}
config.simulation_log_config = {'save_simulation': True, 'log_suffix': 'logging_demo'}

env = SectorIEnv(config=config)

## 3. Run one episode to completion

We take no actions and just step until the episode is `done`/`truncated`.
The auto-save fires at episode end because `save_simulation` is `True`.
(No rendering, so this runs headless.)

In [ ]:
observation_dict, info_dict = env.reset(seed=100)

done = False
steps = 0
while not done and steps < 5000:
    observation_dict, reward_dict, done_dict, truncated_dict, info_dict = env.step({})
    done = all(done_dict.values()) or all(truncated_dict.values())
    steps += 1

print(f'episode finished after {steps} steps')

## 4. Verify the log landed in the correct place

A new `.tar.gz` should have appeared in `LOG_DIR`. If `save_simulation` had not been
enabled, `after - before` would be empty.

In [ ]:
after = {f for f in os.listdir(LOG_DIR) if f.endswith('.tar.gz')} if os.path.isdir(LOG_DIR) else set()
new_files = sorted(after - before)

print('LOG_DIR     :', LOG_DIR)
print('new archives:', new_files)

assert new_files, 'No new .tar.gz was written - logging did not reach LOG_DIR'
for f in new_files:
    full = os.path.join(LOG_DIR, f)
    assert os.path.isfile(full)
    print(f'  {full}  ({os.path.getsize(full)} bytes)')

print('\nOK: gymnasium logs are being written to the bluebird-dt LOG_DIR.')

### Optional: test the env-var override

To confirm `BLUEBIRD_LOG_DIR` redirects the output, set the variable **before** importing
`bluebird_dt` (it is resolved at import time) and re-run the notebook from a fresh kernel:

```python
import os
os.environ['BLUEBIRD_LOG_DIR'] = '/tmp/bluebird_log_test'  # must run before any bluebird_dt import
```